# Portfolio Selection with Limited Holdings on the NASDAQ-100
**IEOR E4004, Group 27E, Milestone 4**

This notebook runs the full model step by step and shows the results.
Run it top to bottom (**Runtime → Run all**). It takes about 5 minutes, mostly for the K = 5 solve.

**Model.** Choose weights for up to K stocks to maximize expected daily return minus λ × mean absolute deviation (MAD). Weights are long-only, sum to 1, and each held stock gets between 1% and 30%. This is a mixed-integer linear program solved with Gurobi.

## 1. Setup
Download the code and data from GitHub and install Gurobi. The free pip license is enough for this model size.

In [ ]:
!git clone -q https://github.com/PhonthakornPhreuksa/ieor4004-group27e.git
%cd ieor4004-group27e
!pip install -q gurobipy yfinance

In [ ]:
import sys
sys.path.append("src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from data import load
from model import solve_mad
from evaluate import buy_and_hold, metrics
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

LAMBDA = 1.0               # baseline, fixed before looking at the test period
K_VALUES = [5, 10, 20, 50, None]   # None = no holdings limit
W_MIN, W_MAX = 0.01, 0.30

### The Gurobi model
All the optimization happens in `src/model.py`, shown below. The notebook calls it through `solve_mad(...)`.

In [ ]:
import gurobipy as gp
print("Gurobi version:", ".".join(map(str, gp.gurobi.version())))
print(open("src/model.py").read())

## 2. Data
- **Universe:** NASDAQ-100 stocks with a complete price history (94 stocks). GOOG is dropped because GOOGL is the same company.
- **Estimation window:** Oct 2, 2023 to Sep 30, 2025 (used to fit the model).
- **Test window:** Oct 1, 2025 to Sep 30, 2026 (used only to evaluate).

In [ ]:
D = load()
R = D["est"].values                 # daily returns, estimation window (T x n)
tickers = np.array(D["universe"])
print("Stocks in universe:", len(tickers))
print("Excluded (incomplete history):", D["excluded"])
print("Estimation days:", len(D["est"]), "| Test days:", len(D["test"]))

### Estimated parameters
µ = mean daily return (shown annualized, × 252). MAD = average daily distance from the mean (a risk measure).

In [ ]:
mu = R.mean(axis=0)
mad = np.abs(R - mu).mean(axis=0)
params = pd.DataFrame({"mean return (annual %)": 100 * mu * 252, "daily MAD (%)": 100 * mad}, index=tickers)
display(params.describe().loc[["min", "50%", "max"]].rename(index={"50%": "median"}))
print("Highest expected return:"); display(params.sort_values("mean return (annual %)", ascending=False).head(5))
print("Lowest risk (MAD):");       display(params.sort_values("daily MAD (%)").head(5))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(params["daily MAD (%)"], params["mean return (annual %)"], s=18, alpha=0.7)
for t in params.sort_values("mean return (annual %)", ascending=False).head(5).index.tolist() + params.sort_values("daily MAD (%)").head(3).index.tolist():
    ax.annotate(t, (params.loc[t, "daily MAD (%)"], params.loc[t, "mean return (annual %)"]), fontsize=8)
ax.set_xlabel("Daily MAD (%)"); ax.set_ylabel("Mean return, annualized (%)")
ax.set_title("Risk vs return of each stock (estimation window)"); ax.grid(alpha=0.3)
plt.show()

## 3. Solve the model for each holdings limit K
Same λ = 1 every time, only K changes. "No limit" removes the holdings constraint but keeps everything else.

### One solve with the full Gurobi log
K = 20 with `verbose=True`, so you can see Gurobi's branch-and-bound progress and the optimality gap.

In [ ]:
_ = solve_mad(R, LAMBDA, K=20, w_min=W_MIN, w_max=W_MAX, verbose=True)

In [ ]:
solutions = {}
for K in K_VALUES:
    label = "No limit" if K is None else f"K={K}"
    solutions[label] = solve_mad(R, LAMBDA, K=K, w_min=W_MIN, w_max=W_MAX, time_limit=300)
    print(f"{label:9s} solved in {solutions[label]['runtime']:6.1f}s, holds {solutions[label]['n_hold']} stocks")

### In-sample results
**Cost** = how much lower the objective is than with no limit, in percent.

In [ ]:
base = solutions["No limit"]["objective"]
insample = pd.DataFrame({
    label: {"holdings": s["n_hold"],
            "objective (daily)": s["objective"],
            "cost vs no limit (%)": 100 * (base - s["objective"]) / abs(base),
            "expected return (annual %)": 100 * s["exp_return"] * 252,
            "daily MAD (%)": 100 * s["mad"],
            "runtime (s)": s["runtime"],
            "proven optimal": s["optimal"]}
    for label, s in solutions.items()}).T
display(insample)

In [ ]:
labels = ["K=5", "K=10", "K=20", "No limit"]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(labels, [insample.loc[l, "cost vs no limit (%)"] for l in labels], color="#007F86")
for i, l in enumerate(labels):
    ax.text(i, insample.loc[l, "cost vs no limit (%)"] + 0.5, f"{insample.loc[l, 'cost vs no limit (%)']:.2f}%", ha="center", fontsize=9)
ax.set_ylabel("Objective lost vs no limit (%)")
ax.set_title("In-sample cost of holding fewer stocks (λ = 1)")
plt.show()

### Which stocks each portfolio holds

In [ ]:
W = pd.DataFrame({l: s["weights"] for l, s in solutions.items()}, index=tickers)
held = W[(W > 0).any(axis=1)].sort_values("K=5", ascending=False)
display((held * 100).round(1).replace(0.0, ""))

## 4. How many stocks does the model want? (no limit, varying λ)
A limit K only matters if the model would hold more than K stocks without it.

In [ ]:
rows = []
for lam in [0, 0.25, 0.5, 1, 2, 5, 10]:
    s = solve_mad(R, lam, K=None, w_min=W_MIN, w_max=W_MAX)
    rows.append({"lambda": lam, "stocks held": s["n_hold"],
                 "expected return (annual %)": 100 * s["exp_return"] * 252,
                 "daily MAD (%)": 100 * s["mad"]})
by_lambda = pd.DataFrame(rows).set_index("lambda")
display(by_lambda)

## 5. Check: is the integer program worth it?
Shortcut: keep the K largest weights from the no-limit portfolio and re-optimize on those K stocks. The MILP searches all combinations, so it should do at least as well.

In [ ]:
free = solutions["No limit"]["weights"]
rows = []
for K in [5, 10, 20]:
    top = np.argsort(-free)[:K]
    shortcut = solve_mad(R[:, top], LAMBDA, K=None, w_min=W_MIN, w_max=W_MAX)["objective"]
    milp = solutions[f"K={K}"]["objective"]
    rows.append({"K": K, "MILP objective": milp, "shortcut objective": shortcut,
                 "MILP better by (%)": 100 * (milp - shortcut) / abs(shortcut)})
display(pd.DataFrame(rows).set_index("K"))

## 6. Out-of-sample test (buy-and-hold)
Each portfolio is bought on Oct 1, 2025 and held for one year with no rebalancing, so weights drift with prices. Compared with QQQ and an equal-weight portfolio of all 94 stocks. Sharpe uses the 3-month Treasury rate (FRED DGS3MO).

In [ ]:
rf = pd.read_csv("data/dgs3mo.csv", index_col=0, parse_dates=True).iloc[:, 0]
rf_daily = rf / 100 / 252

paths = {l: buy_and_hold(D["test"], s["weights"]) for l, s in solutions.items()}
paths["Equal weight"] = buy_and_hold(D["test"], np.full(len(tickers), 1 / len(tickers)))
paths["QQQ"] = (1 + D["qqq_test"]).cumprod()

oos = pd.DataFrame({l: metrics(v, rf_daily) for l, v in paths.items()}).T
oos_pct = oos.copy()
for c in ["total_return", "ann_return", "ann_vol", "max_drawdown", "daily_mad"]:
    oos_pct[c] = 100 * oos[c]
display(oos_pct.rename(columns={"total_return": "total return (%)", "ann_return": "annual return (%)",
                                "ann_vol": "annual volatility (%)", "sharpe": "Sharpe",
                                "max_drawdown": "max drawdown (%)", "daily_mad": "daily MAD (%)"}))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for l, v in paths.items():
    if l == "K=50":      # identical to "No limit"
        continue
    ax.plot(v.index, v.values, "--" if l in ("QQQ", "Equal weight") else "-", lw=1.4, label=l)
ax.set_ylabel("Value of $1 (buy-and-hold)")
ax.set_title("Out-of-sample, Oct 2025 to Sep 2026 (λ = 1)")
ax.legend(ncol=3, fontsize=8, frameon=False); ax.grid(alpha=0.3)
plt.show()

## Takeaways
- Holding 20 stocks costs almost nothing in sample (0.25%). Holding 10 costs 5.8% and holding 5 costs 24%.
- With no limit the model holds 25 stocks, so K = 50 never binds.
- The MILP beats the "keep the biggest" shortcut, most of all when K is small.
- Out of sample, the optimized portfolios had about half the volatility of QQQ but missed most of a strong rally. This is one λ only. Milestone 5 runs the full λ grid and other splits.